# Session 4 CI/CD · The infrastructure pipeline, worked

Two Cloud Storage buckets delivered through plan, policy, approval, apply and drift detection on GitHub Actions.

In [ ]:
source ~/dsba6190-live-demo-04-cicd/env.sh && cd "$WORKDIR" && echo "$PROJECT" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The repository, and who it trusts

In [ ]:
gh variable list

In [ ]:
gcloud iam workload-identity-pools providers describe github \
  --workload-identity-pool github-actions --location global \
  --project "$PROJECT" --format "value(attributeCondition)"

## Step 2 · A compliant pull request

In [ ]:
gh pr create --base main --head demo/label-cost-center --fill
PR=$(gh pr view demo/label-cost-center --json number --jq .number); echo "pull request $PR"
gh pr view "$PR" --web

## Step 3 · Merge, and meet the gate

In [ ]:
gh pr merge "${PR:?}" --squash --delete-branch

In [ ]:
gh run view "$(gh run list --workflow apply --limit 1 --json databaseId --jq '.[0].databaseId')" --web

In [ ]:
gcloud storage buckets describe "gs://$SCRATCH_BUCKET" --format "yaml(labels)"

## Step 4 · A public bucket

In [ ]:
gh pr create --base main --head demo/make-scratch-public --fill
PR=$(gh pr view demo/make-scratch-public --json number --jq .number); echo "pull request $PR"
gh pr view "$PR" --web

In [ ]:
gh pr close "${PR:?}"

## Step 5 · The replace nobody announced

In [ ]:
gh pr create --base main --head demo/move-scratch --fill
PR=$(gh pr view demo/move-scratch --json number --jq .number); echo "pull request $PR"
gh pr view "$PR" --web

In [ ]:
gh pr close "${PR:?}"

## Step 6 · Drift

In [ ]:
# The same label edit the runbook makes in the Console
gcloud storage buckets update "gs://$SCRATCH_BUCKET" --update-labels env=prod-by-accident --project "$PROJECT"

In [ ]:
gh workflow run drift

In [ ]:
gh issue list --label drift --state all --limit 3

In [ ]:
gcloud storage buckets update "gs://$SCRATCH_BUCKET" --update-labels env=dev --project "$PROJECT"
gh workflow run drift

In [ ]:
gh issue list --label drift --state all --limit 3

## After class

In [ ]:
cd "${WORKDIR:?}"
git checkout main && git pull
sed -i '' '/prevent_destroy = true/s/true/false/' main.tf

In [ ]:
terraform init -reconfigure \
  -backend-config="bucket=${STATE_BUCKET:?}" \
  -backend-config="prefix=${STATE_PREFIX:?}"

In [ ]:
cd "${WORKDIR:?}" && terraform destroy -auto-approve

In [ ]:
git checkout -- main.tf